# Data Preprocessing

## Download Dataset

In [ ]:
!pip -q install librosa soundfile numpy scipy pandas pyarrow tqdm

!wget -O groove-v1.0.0.zip https://storage.googleapis.com/magentadata/datasets/groove/groove-v1.0.0.zip
!unzip -q groove-v1.0.0.zip -d /content/gmd
!ls /content/gmd/groove
!rm -f groove-v1.0.0.zip

DATA_DIR = "/content/gmd/groove"
OUT_DIR  = "/content/features"
!mkdir -p $OUT_DIR

import glob
wav_files = glob.glob(f"{DATA_DIR}/**/*.wav", recursive=True)

--2025-11-04 22:19:26--  https://storage.googleapis.com/magentadata/datasets/groove/groove-v1.0.0.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 108.177.98.207, 74.125.135.207, 142.251.188.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|108.177.98.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5111599714 (4.8G) [application/zip]
Saving to: ‘groove-v1.0.0.zip’

groove-v1.0.0.zip   100%[===================>]   4.76G   162MB/s    in 38s     

2025-11-04 22:20:04 (127 MB/s) - ‘groove-v1.0.0.zip’ saved [5111599714/5111599714]

replace /content/gmd/groove/drummer8/session2/12_funk_81_beat_4-4.mid? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/gmd/groove/drummer8/session2/25_latin_84_beat_4-4.mid? [y]es, [n]o, [A]ll, [N]one, [r]ename: nA
replace /content/gmd/groove/drummer8/session2/2_funk_92_beat_4-4.mid? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
n
n
n


drummer1   drummer2  drummer4  drummer6  drummer8  Icon   

## Remove Unnecessary Features

In [ ]:
!pip -q install librosa soundfile pyarrow tqdm

import os, pandas as pd, numpy as np
from tqdm import tqdm
import librosa

OUT_PATH = os.path.join(OUT_DIR, "gmd_features.parquet")

data = pd.read_csv('/content/gmd/groove/info.csv')
df = pd.DataFrame(data)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1150 entries, 0 to 1149
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   drummer         1150 non-null   object 
 1   session         1150 non-null   object 
 2   id              1150 non-null   object 
 3   style           1150 non-null   object 
 4   bpm             1150 non-null   int64  
 5   beat_type       1150 non-null   object 
 6   time_signature  1150 non-null   object 
 7   midi_filename   1150 non-null   object 
 8   audio_filename  1090 non-null   object 
 9   duration        1150 non-null   float64
 10  split           1150 non-null   object 
dtypes: float64(1), int64(1), object(9)
memory usage: 99.0+ KB


In [ ]:
df = df[df["audio_filename"].notna()].copy()

# build absolute paths safely
df["audio_path"] = df["audio_filename"].apply(lambda p: os.path.join(DATA_DIR, str(p)))
df["exists"] = df["audio_path"].apply(os.path.exists)
df = df[df["exists"]].copy()

# Style fields
df["style"] = df["style"].astype(str)
df["style_main"] = df["style"].str.split("/").str[0]   # "funk/groove1" -> "funk"
cleaned_df = df[["audio_path", "style_main"]]
cleaned_df.head()

,audio_path,style_main
0,/content/gmd/groove/drummer1/eval_session/1_fu...,funk
1,/content/gmd/groove/drummer1/eval_session/10_s...,soul
2,/content/gmd/groove/drummer1/eval_session/2_fu...,funk
3,/content/gmd/groove/drummer1/eval_session/3_so...,soul
4,/content/gmd/groove/drummer1/eval_session/4_so...,soul


## Extract Features From Raw Audio Data

In [ ]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

def extract_audio_features(file_path, sr_target=22050):
    """Extract key rhythmic & spectral features from an audio file."""
    result = {"audio_path": file_path}
    try:
        y, sr = librosa.load(file_path, sr=sr_target, mono=True)

        # --- Rhythm ---
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        tempo, _  = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr)
        result["tempo"] = float(tempo)
        result["onset_rate"] = float(np.mean(onset_env))

        # --- Time-domain ---
        result["zcr"] = float(np.mean(librosa.feature.zero_crossing_rate(y)))
        result["rms"] = float(np.mean(librosa.feature.rms(y=y)))

        # --- Spectral features ---
        result["centroid"]  = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
        result["bandwidth"] = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
        result["rolloff"]   = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))

        # --- MFCCs (13 coefficients: mean & std each) ---
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i, val in enumerate(mfcc.mean(axis=1), 1):
            result[f"mfcc{i}_mean"] = float(val)
        for i, val in enumerate(mfcc.std(axis=1), 1):
            result[f"mfcc{i}_std"] = float(val)

    except Exception as e:
        result["error"] = str(e)

    return result

features = []
for path in tqdm(cleaned_df["audio_path"], desc="Extracting features"):
    features.append(extract_audio_features(path))

feat_df = pd.DataFrame(features)
print("Extracted feature rows:", len(feat_df))


Extracting features:   0%|          | 0/1090 [00:00<?, ?it/s]/tmp/ipython-input-3089856605.py:15: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  result["tempo"] = float(tempo)
Extracting features: 100%|██████████| 1090/1090 [10:56<00:00,  1.66it/s]

Extracted feature rows: 1090


In [ ]:
feat_df = feat_df.merge(cleaned_df, on="audio_path", how="left")

feat_df = (
    feat_df
    .drop(columns=["style_main_x", "style_main_y"])
    .rename(columns={"style_main": "style"})
)

print("Target Features", feat_df["style"].unique())
print(feat_df.head())
print(feat_df.info())

Target Features ['funk' 'soul' 'hiphop' 'pop' 'rock' 'jazz' 'neworleans' 'dance' 'latin'
 'afrocuban' 'reggae' 'country' 'gospel' 'punk' 'afrobeat' 'blues'
 'middleeastern']
                                          audio_path       tempo  onset_rate  \
0  /content/gmd/groove/drummer1/eval_session/1_fu...  135.999178    2.128012   
1  /content/gmd/groove/drummer1/eval_session/10_s...   99.384014    2.090060   
2  /content/gmd/groove/drummer1/eval_session/2_fu...  107.666016    2.318351   
3  /content/gmd/groove/drummer1/eval_session/3_so...  112.347147    2.240400   
4  /content/gmd/groove/drummer1/eval_session/4_so...  161.499023    1.923926   

        zcr       rms     centroid    bandwidth      rolloff  mfcc1_mean  \
0  0.235262  0.045458  3551.069656  3025.341735  7147.838124 -295.872162   
1  0.263873  0.038286  3980.431189  2922.163686  7210.735580 -350.589661   
2  0.245070  0.048642  3739.650747  2917.737557  6959.405549 -311.402313   
3  0.203324  0.030960  3538.596986  2976.

## Save Cleaned Dataset

In [ ]:
# Save as Parquet
feat_df.to_parquet("cleaned.parquet", index=False)

# Final Check
df_check = pd.read_parquet("cleaned.parquet")
print(df_check.shape)

(1090, 35)
